# Imports and client init

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import io
import csv
import pandas as pd
from collections import defaultdict
from dotenv import load_dotenv
from datetime import datetime
import requests
import uuid

from linalgo.hub.client import LinalgoClient
from linalgo.annotate.models import Corpus, Document, Annotation, Entity
from wsd.load_data import load_data
from lineval.utils import Body
from linalgo.annotate import models

In [3]:
load_dotenv()
token = os.getenv('LINHUB_TOKEN')
url = "https://linhub.api.linalgo.com/v1"
client = LinalgoClient(token, url)
jack_org = "acf7a1aa-ec18-4fa2-a981-a756bc6e6af2"
test_id = "6667052e-b464-47a9-beca-dd8df8f8c632"

# Creating a task (no issues)

In [7]:
#check existing task data structure
example_task_id = 'd3ce7764-eb85-4999-b965-c028f539ee33'
existing_task = client.get_task(example_task_id, verbose= True)

Retrivieving task with id d3ce7764-eb85-4999-b965-c028f539ee33...
Retrieving annotators... (4 found)
Retrieving entities... (7 found)
Retrieving documents... (63 found)
Retrieving annotations... (431 found)


/home/jack/code/JDryv/linalgo/linalgo-sdk/linalgo/hub/client.py:219: UserWarning: Some annotations have no associated document.
  warnings.warn('Some annotations have no associated document.')


In [8]:
existing_task.__dict__.keys()

dict_keys(['id', 'name', 'description', 'entities', 'corpora', 'annotators', 'annotations', 'documents'])

In [9]:
existing_task.entities[0].__dict__

{'id': '1ecf29a6-9193-4a8b-9317-5dc319b94507', 'name': '3', 'color': '82C9EB'}

In [10]:
#Generate a UID for the task
new_task_id = str(uuid.uuid4())
new_task_id

'49d0af5c-52ee-4e10-83a7-7e2ced42f137'

In [11]:
def create_task(
    name: str,
    organization: str,
    task_id : str = str(uuid.uuid4()),
    description: str = None,
    entities: list[Entity] = [],
    ) -> None:


    serialized_entities = [entity.id for entity in entities]

    post_url = url + f"/tasks/"
    data  = {
        "id": task_id,
        "name": name,
        "slug": name,
        "organization": organization,
        "description": description,
        "entities": serialized_entities,
        "corpora": [corpus.id],
    }
    client.post(url = post_url, data= data)
    pass

In [12]:
# create_task(name = "test_task",
#             organization = jack_org,
#             task_id= new_task_id,
#             entities = existing_task.entities)

In [13]:
# new_task = client.get_task(new_task_id)

In [14]:
# new_task.__dict__

# Load and import Semcor docs

## Loading Semcor

In [15]:
#loading candidates
X, y = load_data(lang='fr')

k = len(X)
X_test, y_test = X[:k], y[:k]
len(X_test), len(y_test)

(24266, 24266)

In [16]:
# canidates to corpus
semcor_corpus = Corpus(name='Semcor')

grouped_X = defaultdict(list)
for i,row in enumerate(X_test):
    row.lemma_meaning = y_test[i]
    grouped_X[(row.lemma, row.pos)].append(row)
items = grouped_X.items()
docs = []
for g, cands in items:
    contexts = "\n".join([anno.context for anno in cands])
    doc = Document(content=contexts,
                   corpus=semcor_corpus
                   )
    doc_annos = []
    for c in cands:
        anno = Annotation(document=doc,
                          entity=c.lemma_meaning,
                          body=Body(text=c.text, context=c.context),
                          task="task",
                          annotator="none",
                          target={},
                          created=datetime.now())
        doc_annos.append(anno)
    doc.annotations = set(doc_annos)
    docs.append(doc)

semcor_corpus.documents = docs
X_docs = semcor_corpus.documents
len(X_docs)

3581

In [17]:
n=0
example_doc = X_docs[n]
example_doc.__dict__.keys()

dict_keys(['id', 'uri', 'content', 'corpus', 'annotations'])

## API call

In [ ]:


organization = client.get_organization('57512466-a2eb-414b-a332-ffd5486a6fb1')
corpus = models.Corpus(name='test', description='Just a test', organization=organization)
corpus = client.create_corpus(corpus, organization)
documents = [
    models.Document(uri=0, content='plop', corpus=corpus),
    models.Document(uri=1, content='plip', corpus=corpus),
    models.Document(uri=2, content='plup', corpus=corpus),
]
client.add_documents(documents)

In [30]:
client.add_documents(X_docs)

<Response [201]>